In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:

import os
import glob
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import matplotlib.pyplot as plt

def remap_mask(mask):
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)
    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val
    return remapped_mask

images_dir = os.path.join(path, "dataset", "images")
masks_dir  = os.path.join(path, "dataset", "masks")

print("Images dir:", images_dir)
print("Masks dir :", masks_dir)

# Collect image files
img_files = sorted([f for f in glob.glob(os.path.join(images_dir, "*")) if os.path.isfile(f)])
mask_files = sorted([f for f in glob.glob(os.path.join(masks_dir, "*")) if os.path.isfile(f)])

mask_map = {os.path.splitext(os.path.basename(m))[0]: m for m in mask_files}

pairs = []
for img in img_files:
    key = os.path.splitext(os.path.basename(img))[0]
    if key in mask_map:
        pairs.append((img, mask_map[key]))

print("Total pairs:", len(pairs))
print("Example pair:", pairs[0])

class SUIMDataset(Dataset):
    def __init__(self, pairs, img_size=(256, 256)):
        self.pairs = pairs
        self.img_size = img_size

        self.img_transform = T.Compose([
            T.Resize(self.img_size),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

        self.mask_resize = T.Resize(self.img_size, interpolation=T.InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        image = Image.open(img_path).convert("RGB")
        mask_img = Image.open(mask_path)

        if mask_img.mode != "L":
            mask_img = mask_img.convert("L")

        image = self.img_transform(image)

        mask_img = self.mask_resize(mask_img)
        mask = torch.from_numpy(np.array(mask_img)).long()

        mask = remap_mask(mask)
        return image, mask

# Create dataset
full_dataset = SUIMDataset(pairs, img_size=(256, 256))

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# Display samples
images, masks = next(iter(train_loader))
plt.figure(figsize=(10, 6))
for i in range(3):
    plt.subplot(3, 2, 2*i + 1)
    img = images[i].cpu()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_vis = (img * std + mean).clamp(0,1)
    plt.imshow(img_vis.permute(1,2,0))
    plt.axis("off")
    plt.title("Image")

    plt.subplot(3, 2, 2*i + 2)
    plt.imshow(masks[i].cpu(), cmap="tab10", vmin=0, vmax=7)
    plt.axis("off")
    plt.title("Mask")
plt.tight_layout()
plt.show()


In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp

num_classes = 8

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes,
    activation=None
)

In [ ]:

import torch.nn as nn
import torch

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for imgs, masks in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            logits = model(imgs)
            loss = criterion(logits, masks)
            total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 10
train_losses, val_losses = [], []

for epoch in range(epochs):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch [{epoch+1}/{epochs}]  Train Loss: {tr_loss:.4f}  Val Loss: {va_loss:.4f}")

plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.show()

In [ ]:
model.eval()
imgs, masks = next(iter(val_loader))
imgs = imgs.to(device)

with torch.no_grad():
    logits = model(imgs)
    preds = torch.argmax(logits, dim=1).cpu()

imgs = imgs.cpu()
masks = masks.cpu()

plt.figure(figsize=(12, 9))
for i in range(3):
    plt.subplot(3, 3, 3*i + 1)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_vis = (imgs[i] * std + mean).clamp(0,1)
    plt.imshow(img_vis.permute(1,2,0))
    plt.axis("off")
    plt.title("Image")

    plt.subplot(3, 3, 3*i + 2)
    plt.imshow(masks[i], cmap="tab10", vmin=0, vmax=7)
    plt.axis("off")
    plt.title("GT Mask")

    plt.subplot(3, 3, 3*i + 3)
    plt.imshow(preds[i], cmap="tab10", vmin=0, vmax=7)
    plt.axis("off")
    plt.title("Pred Mask")

plt.tight_layout()
plt.show()
